In [2]:
import os
import json
import google.generativeai as genai

#

def lookup_order(order_id: str) -> str:
    """Verilmiş sifariş ID-sinə əsasən orders.json faylından məlumatları gətirir."""
    try:
        with open("orders.json", "r") as f:
            orders = json.load(f)
        
        if order_id in orders:
            order = orders[order_id]
            return f"Sifariş {order_id}: Məhsul: {order['item']}, Qiymət: ${order['price']}, Tarix: {order['purchased']}, Zəmanət: {order['warranty_months']} ay."
        else:
            return f"Xəta: {order_id} nömrəli sifariş tapılmadı."
    except Exception as e:
        return f"Fayl oxunarkən xəta baş verdi: {str(e)}"


def calculate(expression: str) -> str:
    """Riyazi ifadələri təhlükəsiz şəkildə hesablayır."""
    try:
        # Təhlükəsizlik üçün giriş mətnini təmizləyirik (yalnız rəqəmlər və riyazi simvollar)
        allowed_chars = "0123456789+-*/(). "
        clean_expression = "".join([c for c in expression if c in allowed_chars])
        return str(eval(clean_expression))
    except Exception as e:
        return f"Hesablama xətası: {str(e)}"

/opt/anaconda3/envs/ironhack/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/v1/2dndwvd55h52x85w8bzl8pmm0000gn/T/ipykernel_1144/1026198506.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
import ollama

def run_tool_loop(user_prompt: str, messages_history: list) -> list:
    """
    Model -> Alət -> Model dövrəsini əllə idarə edən və qısamüddətli
    yaddaşı saxlayan əsas iş axını funksiyası (Lokal Ollama versiyası).
    """
    # 1. İstifadəçinin yeni sorğusunu yaddaşa əlavə edirik (Ollama formatı: content)
    messages_history.append({"role": "user", "content": user_prompt})
    
    # Alət çağırışlarını (tool calling) dəstəkləyən lokal model adı
    # Qeyd: Kompüterinizdə 'ollama pull llama3.1' və ya 'ollama pull qwen2.5' edilmiş olmalıdır
    model_name = 'llama3.1'
    
    max_steps = 5  # Sonsuz dövrənin qarşısını almaq üçün addım limiti
    step = 0
    
    while step < max_steps:
        print(f"\n[ADDIM {step + 1}]: Lokal modelə (Ollama: {model_name}) sorğu göndərilir...")
        
        # Bütün söhbət tarixçəsini modelə və əlimizdəki funksiyaları ötürürük
        # Ollama Python SDK funksiyaları avtomatik JSON sxeminə çevirə bilir
        response = ollama.chat(
            model=model_name,
            messages=messages_history,
            tools=[lookup_order, calculate]
        )
        
        # Modelin qaytardığı köməkçi (assistant) mesajını tarixçəyə əlavə edirik
        messages_history.append(response.message)
        
        # Model hər hansı bir alət (tool) çağırmaq istəyir?
        tool_calls = response.message.tool_calls
        
        if tool_calls:
            for call in tool_calls:
                tool_name = call.function.name
                tool_args = call.function.arguments  # Arqumentlər birbaşa dict olaraq gəlir
                
                print(f" -> [Model Qərarı]: '{tool_name}' alətini çağırmaq lazımdır.")
                print(f" -> [Arqumentlər]: {tool_args}")
                
                # Aləti kod tərəfində əllə (manual) işə salırıq
                if tool_name == "lookup_order":
                    result = lookup_order(**tool_args)
                elif tool_name == "calculate":
                    result = calculate(**tool_args)
                else:
                    result = "Xəta: Naməlum alət."
                
                print(f" -> [Alətin Cavabı]: {result}")
                
                # Alətin nəticəsini standart 'tool' rolu ilə yaddaşa köçürürük
                messages_history.append({
                    "role": "tool",
                    "content": result
                })
            
            step += 1  # Dövrə tamamlandı, növbəti addıma keçirik
        else:
            # Əgər model heç bir alət tələb etməyibsə, deməli son cavabı hazırlayıb
            print(f"\n[YEKUN CAVAB]: {response.message.content}")
            return messages_history

    print("\n[XƏBƏRDARLIQ]: Addım limiti (5) aşıldı! Proses dayandırıldı.")
    return messages_history

In [4]:
def main():

    # Shared memory between turns
    conversation_memory = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. "
                "Use tools when needed. "
                "Remember previous conversation context."
            )
        }
    ]

    print("\n" + "=" * 60)
    print("TURN 1")
    print("=" * 60)

    run_tool_loop(
        "What did order A1001 cost?",
        conversation_memory
    )

    print("\n" + "=" * 60)
    print("TURN 2")
    print("=" * 60)

    run_tool_loop(
        "And what about three of them?",
        conversation_memory
    )

    print("\n" + "=" * 60)
    print("MEMORY CONTENT")
    print("=" * 60)

    for i, message in enumerate(conversation_memory, start=1):
        print(f"\n[{i}]")
        print(message)


if __name__ == "__main__":
    main()


TURN 1

[ADDIM 1]: Lokal modelə (Ollama: llama3.1) sorğu göndərilir...
 -> [Model Qərarı]: 'lookup_order' alətini çağırmaq lazımdır.
 -> [Arqumentlər]: {'order_id': 'A1001'}
 -> [Alətin Cavabı]: Sifariş A1001: Məhsul: laptop, Qiymət: $1200, Tarix: 2026-05-20, Zəmanət: 12 ay.

[ADDIM 2]: Lokal modelə (Ollama: llama3.1) sorğu göndərilir...

[YEKUN CAVAB]: Based on the tool's output, the cost of order A1001 was $1200.

TURN 2

[ADDIM 1]: Lokal modelə (Ollama: llama3.1) sorğu göndərilir...
 -> [Model Qərarı]: 'calculate' alətini çağırmaq lazımdır.
 -> [Arqumentlər]: {'expression': '3 * 1200'}
 -> [Alətin Cavabı]: 3600

[ADDIM 2]: Lokal modelə (Ollama: llama3.1) sorğu göndərilir...

[YEKUN CAVAB]: The total cost for three orders A1001 would be $3600.

MEMORY CONTENT

[1]
{'role': 'system', 'content': 'You are a helpful assistant. Use tools when needed. Remember previous conversation context.'}

[2]
{'role': 'user', 'content': 'What did order A1001 cost?'}

[3]
role='assistant' content='' t